In [2]:
import pandas as pd
import spacy
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
nlp = spacy.load("en_core_web_sm")

In [3]:
content = "Trinamool Congress leader Mahua Moitra has moved the Supreme Court against her expulsion from the Lok Sabha over the cash-for-query allegations against her. Moitra was ousted from the Parliament last week after the Ethics Committee of the Lok Sabha found her guilty of jeopardising national security by sharing her parliamentary portal's login credentials with businessman Darshan Hiranandani."


In [4]:
doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Trinamool Congress 0 18 ORG
Mahua Moitra 26 38 PERSON
the Supreme Court 49 66 ORG
Moitra 157 163 NORP
Parliament 184 194 ORG
last week 195 204 DATE
the Ethics Committee 211 231 ORG
Darshan Hiranandani 373 392 PERSON


In [5]:
text = "TextBlob is very amazing and simple to use. What a great tool!"


In [6]:
blob = TextBlob(text)
blob.sentiment

Sentiment(polarity=0.5933333333333334, subjectivity=0.7023809523809524)

In [8]:
sid_obj = SentimentIntensityAnalyzer()
sentiment_dict = sid_obj.polarity_scores(text)
sentiment_dict

{'neg': 0.0, 'neu': 0.541, 'pos': 0.459, 'compound': 0.8585}

## BERT & GPT

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from transformers import Trainer, TrainingArguments


tokenizer = BertTokenizer.from_pretrained("bert_base_uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased")


In [ ]:
df = pd.read_csv("../../data/IMDB Dataset.csv")
df = df.sample(500)
df['label'] = df['setiment'].map({})
dataset = Dataset.from_pandas(df[['review', 'label']])

def tokenize(batch):
    return tokenizer(batch['review'], truncation=True, padding=max_length, max_length=256)

dataset = dataset.map(tokenize, batched=True)
dataset.set_format (type='torch', columns=['input_ids', 'attention_masks','label'])


In [ ]:
training_args = TrainingArguments(
    output_dir='',
    num_train_epochs=1,
    per_device_train_batch_size=8
)

trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=dataset
)

trainer.train()

In [ ]:
from transformers import  AutoTokenizer, AutoModelForCausalLM

model_name='gpt2'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [ ]:
prompt = ''
 
inputs = tokenizer(prompt, return_tensors='pt').to(device)

ouput = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=1.,
    top_k=50,
    top_p=0.9,
    num_return_sequences=3
)
output = tokenizer.decoder(output[0], skip_special_tokens=True)


## lstm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

with open('..') as f:
    text = f.read().lower()[:20000]

In [ ]:
words  = text.lower().split()

vocab = sorted(set(words))
stoi = {w:i for i,w in enumerate(vocab)}
vocab_size = len(vocab)

data = torch.tensor([stoi[w] for w in words])

seq_len = 10
X, Y = [],[]

for i in range(len(data)-seq_len):
    X.append(data[i:i+seq_len])
    Y.append(data[i+1: i+seq_len+1])

X = torch.stack(X)
Y = torch.stack(Y)

loader = DataLoader(TensorDataset(X, Y), batch_size=32)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.embed = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        x = self.embed(x)
        out,_ = self.lstm(x)
        return self.fc(out)
    
model = LSTMModel()
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)


for X,Y in loader:
    opt.zero_grad()
    out = model(X)
    loss = loss_fn(out, Y)
    loss.backward()
    opt.step()